In [1]:
from Parts.imports import *
from Parts.data_loader import DatasetLoader, DatasetLoaderV2
from Parts.models import U_NET_VANILLA, U_NET_RESNET, U_NET_RESNET_ATTENTION, U_NET_PLUS_PLUS, TRANS_U_NET
from Parts.training_loop import TrainingLoop, TrainingLoopAdvanced, SaveState, EarlyStopping
from Parts.losses import DiceLoss, Mixed_Dice_Sigmoid

DEBUG = False

In [2]:
pipe = albumentations.Compose([
    albumentations.OneOf([
        albumentations.ElasticTransform(p=1),
        albumentations.GridDistortion(p=1)
    ]),
    albumentations.CLAHE(),
    albumentations.RandomBrightnessContrast()
])

In [6]:
image_dir = fr"C:\Users\itizs\Downloads\archive(1)\brisc2025\segmentation_task\train\images"
label_dir = fr"C:\Users\itizs\Downloads\archive(1)\brisc2025\segmentation_task\train\masks"

class_instance : DatasetLoader = DatasetLoaderV2(image_dir, label_dir, (256, 256), pipe)
dataset_loader = DataLoader(class_instance,
                            batch_size=1, shuffle=True)

In [ ]:
if DEBUG :
    image, mask = next(iter(dataset_loader))
    plt.imshow(mask[0].permute(1, 2, 0))

In [ ]:
model = U_NET_VANILLA()
optm = torch.optim.Adam(model.parameters())
loss = Mixed_Dice_Sigmoid()
# well the dice loss is causing problems - loss isn't stably and reliably reducing, its, hence mix of bce and dice to be used


In [ ]:
trainer = TrainingLoop(model, loss, optm, 6)
trainer.train(dataset_loader)

In [ ]:
early_stopper = EarlyStopping()
saver = SaveState('SimpleUNet', False)

trainier = TrainingLoopAdvanced(model, loss, optm, 1, None, saver)

In [ ]:
# from torch.utils.data import Subset # <----------- Such convinient thing they have
# subset_class_instance = Subset(class_instance, [0, 1, 2])
# subset_test_data_loader = DataLoader(subset_class_instance, 1)

# val_subset = Subset(class_instance, [3])
# val_subset_loader = DataLoader(val_subset, 1)

In [ ]:
# trainier.train(subset_test_data_loader, val_subset_loader)

In [7]:
# well let us see class imbalance
empty_count, non_empty = 0, 0

total = len(dataset_loader)
with tqdm.tqdm(total=total) as bar:
    for data, label in dataset_loader:
        if label.cpu().detach().numpy().unsqueeze().sum() == 0:
            empty_count += 1
        else:
            non_empty += 1
        bar.update(1)

100%|██████████| 3933/3933 [02:36<00:00, 25.12it/s]


In [8]:
print(f"empty :: {empty_count} :: Non empty :: {non_empty}")

empty :: 0 :: Non empty :: 3933
